# 🤖 Z.A.I.N.E — 1-Click QLoRA Fine-Tuning on Google Colab

> **Target Model:** `Qwen/Qwen2.5-3B-Instruct`  
> **Runtime Required:** Free Google Colab T4 GPU (16 GB VRAM)  
> **Training Time:** ~10–15 minutes  

### ⚡ Instructions:
1. In Google Colab menu, click **Runtime** → **Change runtime type** → select **T4 GPU** → **Save**.
2. In the menu, click **Runtime** → **Run all** (or run each cell sequentially).
3. When prompted in Cell 2, upload your `final_dataset.jsonl` from your `Project-Z/finetune/` folder.
4. At the end, your browser will automatically download `zaine-qlora-adapter.zip`!

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!nvidia-smi
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

### Step 2: Upload Dataset (`final_dataset.jsonl`)

In [ ]:
import os
from google.colab import files

if not os.path.exists('final_dataset.jsonl'):
    print('Please upload final_dataset.jsonl from your Project-Z/finetune/ folder:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'final_dataset.jsonl':
            os.rename(fn, 'final_dataset.jsonl')
else:
    print('final_dataset.jsonl already exists!')

with open('final_dataset.jsonl', 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(f'Dataset loaded successfully with {len(lines)} training examples.')

### Step 3: Run QLoRA Fine-Tuning

In [ ]:
import inspect
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DATASET_PATH = "final_dataset.jsonl"
OUTPUT_DIR = "./zaine-qlora-adapter"
MAX_SEQ_LENGTH = 1024

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

print(f"Loading tokenizer for {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Formatting dataset with chat template...")
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print(f"Prepared {len(dataset)} examples for training.")

print("Loading base model in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

print("Configuring LoRA adapter...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

sft_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "logging_steps": 5,
    "save_strategy": "epoch",
    "bf16": True,
    "report_to": "none",
}

sft_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sft_params:
    sft_kwargs["max_length"] = MAX_SEQ_LENGTH
elif "max_seq_length" in sft_params:
    sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

if "dataset_text_field" in sft_params:
    sft_kwargs["dataset_text_field"] = "text"

sft_config = SFTConfig(**sft_kwargs)

trainer_kwargs = {
    "model": model,
    "args": sft_config,
    "train_dataset": dataset,
}

trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" not in sft_params and "dataset_text_field" in trainer_params:
    trainer_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**trainer_kwargs)

print("\n🚀 Training started! Please wait...")
trainer.train()

print(f"\nSaving trained adapter to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Training complete!")

### Step 4: Zip & Download Trained Adapter

In [ ]:
import shutil
from google.colab import files

zip_filename = "zaine-qlora-adapter.zip"
print(f"Zipping {OUTPUT_DIR} into {zip_filename}...")
shutil.make_archive("zaine-qlora-adapter", 'zip', OUTPUT_DIR)

print("Downloading adapter to your computer...")
files.download(zip_filename)
print("Done! Check your browser downloads folder.")

### Step 5: What to do after downloading:
1. Extract `zaine-qlora-adapter.zip` into `Project-Z/finetune/zaine-3b-qlora/` on your computer.
2. In your local terminal, run:
   ```bash
   python merge_adapter.py
   ollama create zaine -f Modelfile
   ```
3. Run `python main.py` to talk with your newly fine-tuned Z.A.I.N.E model!